In [2]:
from datasets import load_dataset

#loading from hugging face
dataset = load_dataset("tattabio/ec_classification")
## making test and train set
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
##sanity check to make sure dims are the same as on the DGEB paper dimensions
print(train_df.shape)
print(test_df.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/285k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/87.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/512 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

(512, 3)
(128, 3)


In [6]:
train_df.head()
## checking what the label and seq looks like

,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...


In [7]:
x_train_seq = train_df["Sequence"].tolist()
y_train_labels = train_df["Label"].tolist()

x_test_seq = test_df["Sequence"].tolist()
y_test_labels = test_df["Label"].tolist()

from sklearn.neural_network import MLPClassifier
## defining model
model = MLPClassifier(max_iter=1000)

In [9]:
!pip install dgeb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 15.0 MB/s eta 0:00:00


In [10]:
import dgeb
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
## first input which is just the embeddings i am using dgeb embedding model to make each seq into a vector
embedding_model = dgeb.get_model("facebook/esm2_t6_8M_UR50D")

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [11]:
## converting train data to numerical vectors
x_train_input1 = embedding_model.encode(x_train_seq).mean(axis=1)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
encoding:   0%|          | 0/4 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationa

In [12]:
## doing same for test data
## broke into two cells so we would avoid rerunning

import numpy as np
x_test_p1 = embedding_model.encode(x_test_seq[:127]).mean(axis=1)
last_test = embedding_model.encode(x_test_seq[127:]).mean(axis=1)
x_test_input1 = np.vstack([x_test_p1, last_test])

In [13]:
from sklearn.metrics import accuracy_score, f1_score
## checking shape
print(x_train_input1.shape)
print(x_test_input1.shape)
## now fitting model on this info
model.fit(x_train_input1, y_train_labels)
predictions_input1 = model.predict(x_test_input1)

## accuracy and f1 score for input 1
print("accuracy:", accuracy_score(y_test_labels, predictions_input1))
print("f1:", f1_score(y_test_labels, predictions_input1, average="macro"))

(512, 320)
(128, 320)
accuracy: 0.453125
f1: 0.40286458333333336


In [14]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score
## now doing input 2 which is k mer counts of 3

## defining a function to make kmer
def kmerFunction(seq):
    return " ".join([seq[i:i+3] for i in range(len(seq)-2)])


train_kmer = [kmerFunction(seq) for seq in x_train_seq]
test_kmer = [kmerFunction(seq) for seq in x_test_seq]

## now counting how many times each kmer shows up
helper = CountVectorizer()
x_train_input2 = helper.fit_transform(train_kmer)
x_test_input2 = helper.transform(test_kmer)

## checking shape
print(x_train_input2.shape)
print(x_test_input2.shape)

## defining model again
model2 = MLPClassifier(max_iter=1000)
model2.fit(x_train_input2, y_train_labels)
predictions_input2 = model2.predict(x_test_input2)

## printing accuracy and f1 score
print("accuracy:", accuracy_score(y_test_labels, predictions_input2))
print("f1:", f1_score(y_test_labels, predictions_input2, average="macro"))

(512, 7986)
(128, 7986)
accuracy: 0.1171875
f1: 0.09129464285714285


In [15]:
## now doing input 3 which is weighted kmer
from sklearn.feature_extraction.text import TfidfVectorizer

## defining the weighted helper
weighted = TfidfVectorizer()

x_train_input3 = weighted.fit_transform(train_kmer)
x_test_input3 = weighted.transform(test_kmer)

## ensuring shape is accurate
print(x_train_input3.shape)
print(x_test_input3.shape)

## making model3
model3 = MLPClassifier(max_iter=1000)

model3.fit(x_train_input3, y_train_labels)
predictions_input3 = model3.predict(x_test_input3)

## f1 and accuracy scores
print("accuracy:", accuracy_score(y_test_labels, predictions_input3))
print("f1:", f1_score(y_test_labels, predictions_input3, average="macro"))

(512, 7986)
(128, 7986)
accuracy: 0.1953125
f1: 0.15341546474358975
